# Distributed Reconciliation with Dask

> Using Dask arrays for distributed hierarchical reconciliation

The `HierarchicalForecast` package supports distributed computation using [Dask arrays](https://docs.dask.org/en/stable/array.html). This can be particularly useful when working with large hierarchies where the matrix operations involved in reconciliation can benefit from parallelization.

In this notebook, we will:
1. Load a hierarchical dataset
2. Generate base forecasts
3. Compare reconciliation results with and without distributed computation
4. Benchmark the runtime performance of both approaches

You can run these experiments using CPU or GPU with Google Colab.

<a href="https://colab.research.google.com/github/Nixtla/hierarchicalforecast/blob/main/nbs/examples/distributedreconciliation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Libraries

In [1]:
# Install required packages
# !pip install hierarchicalforecast[distributed] statsforecast datasetsforecast

In [2]:
import time
import warnings

import numpy as np
import pandas as pd

from datasetsforecast.hierarchical import HierarchicalData, HierarchicalInfo

warnings.filterwarnings('ignore')

## 2. Load Data

We'll use the `TourismSmall` dataset which contains a hierarchical structure with multiple levels:
- Country (top level)
- Country/Purpose
- Country/Purpose/State
- Country/Purpose/State/CityNonCity (bottom level)

In [3]:
group_name = 'TourismSmall'
group = HierarchicalInfo.get_group(group_name)
Y_df, S_df, tags = HierarchicalData.load('./data', group_name)
S_df = S_df.reset_index(names="unique_id")
Y_df['ds'] = pd.to_datetime(Y_df['ds'])

In [4]:
print(f"Number of series: {Y_df['unique_id'].nunique()}")
print(f"Number of bottom-level series: {len(tags[list(tags.keys())[-1]])}")
print(f"Hierarchy levels: {list(tags.keys())}")

Number of series: 89
Number of bottom-level series: 56
Hierarchy levels: ['Country', 'Country/Purpose', 'Country/Purpose/State', 'Country/Purpose/State/CityNonCity']


In [5]:
# Show the summing matrix structure
print(f"Summing matrix shape: {S_df.shape}")
S_df.head()

Summing matrix shape: (89, 57)


,unique_id,nsw-hol-city,nsw-hol-noncity,vic-hol-city,vic-hol-noncity,qld-hol-city,qld-hol-noncity,sa-hol-city,sa-hol-noncity,wa-hol-city,...,qld-oth-city,qld-oth-noncity,sa-oth-city,sa-oth-noncity,wa-oth-city,wa-oth-noncity,tas-oth-city,tas-oth-noncity,nt-oth-city,nt-oth-noncity
0,total,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,hol,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,vfr,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,bus,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,oth,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [6]:
# Split into train/test
Y_test_df = Y_df.groupby('unique_id').tail(group.horizon)
Y_train_df = Y_df.drop(Y_test_df.index)

print(f"Training periods: {Y_train_df.groupby('unique_id').size().iloc[0]}")
print(f"Test periods (horizon): {group.horizon}")

Training periods: 32
Test periods (horizon): 4


## 3. Generate Base Forecasts

We'll use `StatsForecast` to generate base forecasts using AutoARIMA and Naive models.

In [7]:
from statsforecast.core import StatsForecast
from statsforecast.models import AutoARIMA, Naive

In [8]:
fcst = StatsForecast(
    models=[AutoARIMA(season_length=group.seasonality), Naive()], 
    freq="QE", 
    n_jobs=-1
)
Y_hat_df = fcst.forecast(df=Y_train_df, h=group.horizon)
Y_hat_df.head()

,unique_id,ds,AutoARIMA,Naive
0,bus,2006-03-31,8918.478516,11547.0
1,bus,2006-06-30,9581.925781,11547.0
2,bus,2006-09-30,11194.676758,11547.0
3,bus,2006-12-31,10678.958008,11547.0
4,hol,2006-03-31,42805.347656,26418.0


## 4. Hierarchical Reconciliation: Normal vs Distributed

Now we'll compare the reconciliation results using:
1. **Normal mode** (`distributed=False`): Uses standard NumPy arrays
2. **Distributed mode** (`distributed=True`): Converts arrays to Dask arrays for parallel computation

We'll use several reconciliation methods:
- `BottomUp`: Simple aggregation from bottom level
- `TopDown`: Distributes top-level forecasts using proportions
- `MinTrace(method='ols')`: Optimal reconciliation using ordinary least squares
- `MinTrace(method='wls_struct')`: Weighted least squares based on hierarchy structure

In [9]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, TopDown, MinTrace, MiddleOut

In [10]:
# Define reconcilers to test
reconcilers = [
    BottomUp(),
    TopDown(method='forecast_proportions'),
    MinTrace(method='ols'),
    MinTrace(method='wls_struct'),
]

### 4.1 Normal Reconciliation (NumPy)

In [11]:
hrec_normal = HierarchicalReconciliation(reconcilers=reconcilers)

Y_rec_normal = hrec_normal.reconcile(
    Y_hat_df=Y_hat_df, 
    Y_df=Y_train_df, 
    S_df=S_df, 
    tags=tags,
    distributed=False  # Use NumPy arrays (default)
)

print("Normal reconciliation columns:")
print([col for col in Y_rec_normal.columns if 'AutoARIMA' in col])

Normal reconciliation columns:
['AutoARIMA', 'AutoARIMA/BottomUp', 'AutoARIMA/TopDown_method-forecast_proportions', 'AutoARIMA/MinTrace_method-ols', 'AutoARIMA/MinTrace_method-wls_struct']


### 4.2 Distributed Reconciliation (Dask)

In [12]:
hrec_distributed = HierarchicalReconciliation(reconcilers=reconcilers)

Y_rec_distributed = hrec_distributed.reconcile(
    Y_hat_df=Y_hat_df, 
    Y_df=Y_train_df, 
    S_df=S_df, 
    tags=tags,
    distributed=True,  # Use Dask arrays
    chunks="auto"      # Let Dask determine optimal chunk sizes
)

print("Distributed reconciliation columns:")
print([col for col in Y_rec_distributed.columns if 'AutoARIMA' in col])

Distributed reconciliation columns:
['AutoARIMA', 'AutoARIMA/BottomUp', 'AutoARIMA/TopDown_method-forecast_proportions', 'AutoARIMA/MinTrace_method-ols', 'AutoARIMA/MinTrace_method-wls_struct']


### 4.3 Compare Results

Let's verify that both methods produce identical results.

In [13]:
# Get reconciled columns (excluding original forecast columns)
reconciled_cols = [col for col in Y_rec_normal.columns if '/' in col]

print("Comparing reconciliation results between normal and distributed modes:\n")

all_match = True
for col in reconciled_cols:
    normal_values = Y_rec_normal[col].values
    distributed_values = Y_rec_distributed[col].values
    
    # Check if values match (within floating point tolerance)
    max_diff = np.max(np.abs(normal_values - distributed_values))
    match = np.allclose(normal_values, distributed_values, rtol=1e-10, atol=1e-10)
    
    status = "✓ MATCH" if match else "✗ DIFFER"
    print(f"{col}: {status} (max diff: {max_diff:.2e})")
    
    if not match:
        all_match = False

print(f"\n{'='*50}")
if all_match:
    print("All reconciliation methods produce identical results!")
else:
    print("Some methods produced different results.")

Comparing reconciliation results between normal and distributed modes:

AutoARIMA/BottomUp: ✓ MATCH (max diff: 0.00e+00)
Naive/BottomUp: ✓ MATCH (max diff: 0.00e+00)
AutoARIMA/TopDown_method-forecast_proportions: ✓ MATCH (max diff: 0.00e+00)
Naive/TopDown_method-forecast_proportions: ✓ MATCH (max diff: 0.00e+00)
AutoARIMA/MinTrace_method-ols: ✓ MATCH (max diff: 0.00e+00)
Naive/MinTrace_method-ols: ✓ MATCH (max diff: 0.00e+00)
AutoARIMA/MinTrace_method-wls_struct: ✓ MATCH (max diff: 0.00e+00)
Naive/MinTrace_method-wls_struct: ✓ MATCH (max diff: 0.00e+00)

All reconciliation methods produce identical results!


## 5. Runtime Benchmark

Let's compare the runtime performance of normal vs distributed reconciliation. We'll run multiple iterations to get stable timing measurements.

**Note**: For small datasets like TourismSmall, the overhead of Dask may make distributed computation slower. The benefits of distributed computation are more apparent with larger hierarchies.

In [14]:
def benchmark_reconciliation(Y_hat_df, Y_train_df, S_df, tags, reconcilers, 
                              distributed=False, n_runs=5):
    """Benchmark reconciliation runtime."""
    times = []
    
    for i in range(n_runs):
        hrec = HierarchicalReconciliation(reconcilers=reconcilers)
        
        start = time.perf_counter()
        _ = hrec.reconcile(
            Y_hat_df=Y_hat_df, 
            Y_df=Y_train_df, 
            S_df=S_df, 
            tags=tags,
            distributed=distributed
        )
        end = time.perf_counter()
        
        times.append(end - start)
    
    return {
        'mean': np.mean(times),
        'std': np.std(times),
        'min': np.min(times),
        'max': np.max(times),
        'times': times
    }

In [15]:
n_runs = 5

print(f"Running benchmark with {n_runs} iterations each...\n")

# Benchmark normal mode
print("Benchmarking normal mode (NumPy)...")
normal_results = benchmark_reconciliation(
    Y_hat_df, Y_train_df, S_df, tags, reconcilers,
    distributed=False, n_runs=n_runs
)

# Benchmark distributed mode
print("Benchmarking distributed mode (Dask)...")
distributed_results = benchmark_reconciliation(
    Y_hat_df, Y_train_df, S_df, tags, reconcilers,
    distributed=True, n_runs=n_runs
)

print("\nBenchmark complete!")

Running benchmark with 5 iterations each...

Benchmarking normal mode (NumPy)...
Benchmarking distributed mode (Dask)...

Benchmark complete!


In [16]:
# Create results dataframe
benchmark_df = pd.DataFrame({
    'Mode': ['Normal (NumPy)', 'Distributed (Dask)'],
    'Mean (s)': [normal_results['mean'], distributed_results['mean']],
    'Std (s)': [normal_results['std'], distributed_results['std']],
    'Min (s)': [normal_results['min'], distributed_results['min']],
    'Max (s)': [normal_results['max'], distributed_results['max']],
})

print("=" * 70)
print("RUNTIME BENCHMARK RESULTS")
print("=" * 70)
print(f"\nDataset: {group_name}")
print(f"Number of series: {Y_df['unique_id'].nunique()}")
print(f"Number of reconcilers: {len(reconcilers)}")
print(f"Number of runs: {n_runs}")
print("\n")

print(benchmark_df.to_string(index=False))

# Calculate speedup/slowdown
ratio = normal_results['mean'] / distributed_results['mean']
print(f"\nSpeedup ratio (Normal/Distributed): {ratio:.2f}x")

if ratio > 1:
    print(f"Distributed mode is {ratio:.2f}x faster")
else:
    print(f"Normal mode is {1/ratio:.2f}x faster")
    print("\nNote: For small datasets, Dask overhead may exceed parallelization benefits.")
    print("Distributed mode shows advantages with larger hierarchies.")

RUNTIME BENCHMARK RESULTS

Dataset: TourismSmall
Number of series: 89
Number of reconcilers: 4
Number of runs: 5


              Mode  Mean (s)  Std (s)  Min (s)  Max (s)
    Normal (NumPy)  0.058223 0.024955 0.041135 0.106820
Distributed (Dask)  0.080987 0.001824 0.078632 0.084057

Speedup ratio (Normal/Distributed): 0.72x
Normal mode is 1.39x faster

Note: For small datasets, Dask overhead may exceed parallelization benefits.
Distributed mode shows advantages with larger hierarchies.


### 5.1 Per-Reconciler Timing

Let's also look at the execution time for each individual reconciler.

In [17]:
# Get per-reconciler timing from the last run
print("Per-reconciler execution times (from last run):\n")

print("Normal mode (NumPy):")
hrec_normal_timing = HierarchicalReconciliation(reconcilers=reconcilers)
_ = hrec_normal_timing.reconcile(
    Y_hat_df=Y_hat_df, Y_df=Y_train_df, S_df=S_df, tags=tags, distributed=False
)
for name, exec_time in hrec_normal_timing.execution_times.items():
    print(f"  {name}: {exec_time*1000:.2f} ms")

print("\nDistributed mode (Dask):")
hrec_dist_timing = HierarchicalReconciliation(reconcilers=reconcilers)
_ = hrec_dist_timing.reconcile(
    Y_hat_df=Y_hat_df, Y_df=Y_train_df, S_df=S_df, tags=tags, distributed=True
)
for name, exec_time in hrec_dist_timing.execution_times.items():
    print(f"  {name}: {exec_time*1000:.2f} ms")

Per-reconciler execution times (from last run):

Normal mode (NumPy):
  AutoARIMA/BottomUp: 1.44 ms
  Naive/BottomUp: 1.08 ms
  AutoARIMA/TopDown_method-forecast_proportions: 4.80 ms
  Naive/TopDown_method-forecast_proportions: 4.58 ms
  AutoARIMA/MinTrace_method-ols: 1.48 ms
  Naive/MinTrace_method-ols: 1.31 ms
  AutoARIMA/MinTrace_method-wls_struct: 1.30 ms
  Naive/MinTrace_method-wls_struct: 1.41 ms

Distributed mode (Dask):
  AutoARIMA/BottomUp: 4.49 ms
  Naive/BottomUp: 4.35 ms
  AutoARIMA/TopDown_method-forecast_proportions: 7.11 ms
  Naive/TopDown_method-forecast_proportions: 7.42 ms
  AutoARIMA/MinTrace_method-ols: 8.13 ms
  Naive/MinTrace_method-ols: 8.40 ms
  AutoARIMA/MinTrace_method-wls_struct: 7.18 ms
  Naive/MinTrace_method-wls_struct: 7.22 ms


## 6. Using Dask Arrays Directly

You can also use Dask arrays directly with the reconciliation methods, bypassing the `HierarchicalReconciliation` class. This gives you more control over the computation.

In [18]:
import dask.array as da
from hierarchicalforecast.methods import BottomUp, MinTrace

In [19]:
# Prepare NumPy arrays
S_np = S_df.drop(columns=['unique_id']).values.astype(np.float64)

# Get y_hat for a single model
y_hat_np = Y_hat_df.pivot(index='unique_id', columns='ds', values='AutoARIMA').values

# Get bottom level indices
idx_bottom = list(range(S_np.shape[0] - S_np.shape[1], S_np.shape[0]))

print(f"S matrix shape: {S_np.shape}")
print(f"y_hat shape: {y_hat_np.shape}")
print(f"Number of bottom series: {len(idx_bottom)}")

S matrix shape: (89, 56)
y_hat shape: (89, 4)
Number of bottom series: 56


In [20]:
# Convert to Dask arrays
S_dask = da.from_array(S_np, chunks="auto")
y_hat_dask = da.from_array(y_hat_np, chunks="auto")

print(f"S_dask chunks: {S_dask.chunks}")
print(f"y_hat_dask chunks: {y_hat_dask.chunks}")

S_dask chunks: ((89,), (56,))
y_hat_dask chunks: ((89,), (4,))


In [21]:
# Use BottomUp with Dask arrays
bottom_up = BottomUp()
result_dask = bottom_up(S=S_dask, y_hat=y_hat_dask, idx_bottom=idx_bottom)

print("BottomUp result with Dask arrays:")
print(f"Mean shape: {result_dask['mean'].shape}")
print(f"First few values: {result_dask['mean'][:3, :]}")

BottomUp result with Dask arrays:
Mean shape: (89, 4)
First few values: [[196302.35673523 145274.03378296 151400.07685852 151341.36952209]
 [ 20900.46199036  19051.63342285  19452.86587524  19570.91413879]
 [ 10787.89048767   6519.38543701   5284.27429199   6508.7399292 ]]


In [22]:
# Verify against NumPy result
result_numpy = bottom_up(S=S_np, y_hat=y_hat_np, idx_bottom=idx_bottom)

print("Verification:")
print(f"Results match: {np.allclose(result_dask['mean'], result_numpy['mean'])}")

Verification:
Results match: True


## 7. Summary

In this notebook, we demonstrated:

1. **Distributed reconciliation** using the `distributed=True` parameter in `HierarchicalReconciliation.reconcile()`

2. **Result equivalence**: Both normal and distributed modes produce identical reconciliation results

3. **Runtime comparison**: For small datasets, the overhead of Dask may outweigh parallelization benefits. However, distributed computation becomes advantageous with larger hierarchies.

4. **Direct Dask usage**: You can use Dask arrays directly with individual reconciliation methods for more control.

### When to use distributed mode:

- **Large hierarchies** with many series (1000+)
- **Long forecast horizons** requiring large matrix operations
- **Memory-constrained environments** where chunked computation is beneficial
- **Cluster computing** where Dask can distribute work across multiple machines

### Installation

To use distributed features, install with:
```bash
pip install hierarchicalforecast[distributed]
```

## References

- [Dask Arrays Documentation](https://docs.dask.org/en/stable/array.html)
- [Hyndman, R.J., & Athanasopoulos, G. (2021). "Forecasting: principles and practice, 3rd edition: Chapter 11: Forecasting hierarchical and grouped series."](https://otexts.com/fpp3/hierarchical.html)
- [Wickramasuriya, S. L., Athanasopoulos, G., & Hyndman, R. J. (2019). Optimal forecast reconciliation for hierarchical and grouped time series through trace minimization.](https://doi.org/10.1080/01621459.2018.1448825)